In [ ]:
repository_filter: list[str] = []
top_n_repos: int = 20

In [ ]:
from moderne_visualizations_misc.reusable.data_loader import read_data_table
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
import warnings

warnings.simplefilter("ignore")

# df = read_data_table("../samples/test_quality_issues.csv")
df = read_data_table("../samples/v2/io.moderne.prethink.table.TestQualityIssues.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)

In [ ]:
if len(df) == 0:
    fig = qu.empty_figure()
else:
    total_repos = df["repoShort"].nunique()
    repo_totals = df.groupby("repoShort").size().nlargest(top_n_repos)
    top_repos = repo_totals.index.tolist()
    elided = total_repos - len(top_repos)
    filtered = df[df["repoShort"].isin(top_repos)]

    issue_types = filtered["issueType"].unique().tolist()

    pivot = (
        filtered.groupby(["repoShort", "issueType"])
        .size()
        .unstack(fill_value=0)
    )
    pivot = pivot.reindex(columns=issue_types, fill_value=0)
    pivot = pivot.loc[repo_totals.index]

    text_vals = [[str(v) for v in row] for row in pivot.values]

    fig = go.Figure(
        data=go.Heatmap(
            z=pivot.values,
            x=pivot.columns.tolist(),
            y=pivot.index.tolist(),
            colorscale=[
                [0, "#FFFFFF"],
                [0.33, "#FFE082"],
                [0.66, "#FF8A65"],
                [1, "#EF5350"],
            ],
            text=text_vals,
            texttemplate="%{text}",
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Issue: %{x}<br>"
                "Count: %{text}"
                "<extra></extra>"
            ),
            colorbar=dict(title="Issue Count"),
        )
    )
    fig.update_layout(
        title="Test Quality Issues by Repository and Type",
        xaxis_title="Issue Type",
        yaxis_title="",
        height=max(500, len(pivot) * 28 + 150),
        margin=dict(l=200, r=50, t=60, b=80),
        yaxis=dict(autorange="reversed"),
    )

    if elided > 0:
        fig.add_annotation(
            text=f"Showing top {len(top_repos)} of {total_repos} repositories. {elided} additional repositories with issues not shown.",
            xref="paper", yref="paper", x=0.5, y=-0.08,
            showarrow=False, font=dict(size=11, color="#888"),
        )

fig.show()